# F5-probability — Session 02: Variance and Independence

**Session length:** about 85 minutes • **Concepts:** variance, independence,
variance-of-sums — how spread is measured, when it adds, and the scaled-sums
identity the exam builds a derivation on.

Checkpoint answers are collected at the end of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Measuring spread: variance

Two games both have expectation 0: game $A$ pays $-1$ or $+1$ (fair coin);
game $B$ pays $-100$ or $+100$ (fair coin). Expectation cannot tell them
apart, but any player can: $B$ is wildly riskier. We need a number for
*spread around the expectation*.

First attempt: average the deviation $X - E[X]$. Session 01's Checkpoint 6
already killed this idea — $E[X - E[X]] = 0$ for *every* random variable;
positive and negative deviations cancel exactly. The standard fix is to
square deviations before averaging:

$$\operatorname{Var}[X] \;=\; E\big[(X - E[X])^2\big]
 \;=\; \sum_x (x - E[X])^2\, P(X = x).$$

**Variance** is the expected squared deviation from the expectation. It is 0
exactly when $X$ is constant, and it grows as outcomes scatter farther from
$E[X]$. For game $A$: $\operatorname{Var} = \tfrac12(-1-0)^2 +
\tfrac12(1-0)^2 = 1$. For game $B$: $10{,}000$.

Because squaring changes units (tokens → tokens²), we also name the square
root: the **standard deviation** $\sigma = \sqrt{\operatorname{Var}[X]}$,
back in the original units. Convention: $\mu = E[X]$,
$\sigma^2 = \operatorname{Var}[X]$.

A worked table computation — the spinner from Session 01
($x = 1, 2, 5$ with $p = \tfrac12, \tfrac14, \tfrac14$, $\mu = 2.25$):

$$\operatorname{Var}[X] = \tfrac12(1 - 2.25)^2 + \tfrac14(2 - 2.25)^2
 + \tfrac14(5 - 2.25)^2 = 0.78125 + 0.015625 + 1.890625 = 2.6875.$$

The same indexed sum in code, and its simulation estimate (the mean of
squared deviations of the samples):

In [ ]:
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])

mu = (values * probs).sum()
var_exact = (((values - mu) ** 2) * probs).sum()     # sum_i p_i (x_i - mu)^2
print("mu:", mu, "  Var exact:", var_exact, "  sigma:", np.sqrt(var_exact))

SEED = 20260804
rng = np.random.default_rng(SEED)
spins = rng.choice(values, size=100_000, p=probs)
dev = spins - spins.mean()
print("Var estimated from samples:", (dev ** 2).mean())

### Checkpoint 1

1. By hand: compute $\operatorname{Var}$ and $\sigma$ for the fair coin
   paying $-1$ or $+1$, and for a rigged coin paying $-1$ with probability
   $0.9$ and $+9$ with probability $0.1$. (Compute each $\mu$ first.)
2. Why is $E[X - E[X]]$ useless as a spread measure while
   $E[|X - E[X]|]$ would at least work? Give the one-line algebra reason for
   the first and compute the second for game $A$.

## 2. The shortcut: Var[X] = E[X²] − E[X]²

Expanding the square inside the definition gives the identity you will use
far more often than the definition itself.

**Claim.** For any random variable $X$ with $\mu = E[X]$:
$\operatorname{Var}[X] = E[X^2] - (E[X])^2$.

**Proof** (discrete — only linearity from Session 01):

$$\operatorname{Var}[X] = E[(X - \mu)^2]
 = E[X^2 - 2\mu X + \mu^2]
 = E[X^2] - 2\mu\,E[X] + \mu^2
 = E[X^2] - 2\mu^2 + \mu^2 = E[X^2] - \mu^2. \;\blacksquare$$

The middle step is pure linearity: $\mu$ is a constant, so
$E[-2\mu X] = -2\mu E[X]$ and $E[\mu^2] = \mu^2$.

Spinner check: Session 01 computed $E[X^2] = 7.75$ and $E[X] = 2.25$, so
$\operatorname{Var}[X] = 7.75 - 5.0625 = 2.6875$ — matching Section 1's
table computation exactly.

The shortcut is also why the Session 01 trap matters so much: $E[X^2]$ and
$(E[X])^2$ *differ*, and variance is precisely the gap between them. Since
variance is a sum of squared terms it can never be negative, which hands you
a bonus fact for free: $E[X^2] \ge (E[X])^2$ always. If your shortcut
computation ever comes out negative, you subtracted in the wrong order.

In [ ]:
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])

e_x = (values * probs).sum()
e_x2 = ((values ** 2) * probs).sum()
print("shortcut:  ", e_x2 - e_x ** 2)

mu = e_x                                              # definition, for comparison
print("definition:", (((values - mu) ** 2) * probs).sum())

### Checkpoint 2

1. By hand with the shortcut: the fair die has $E[X] = 3.5$ and
   $E[X^2] = \tfrac{91}{6}$ (Session 01). Show
   $\operatorname{Var}[X] = \tfrac{35}{12}$.
2. A classmate computes $E[X]^2 - E[X^2]$ for the die, gets
   $-\tfrac{35}{12}$, and reports a negative variance. Which fact from this
   section instantly flags the error?

## 3. Shifting and scaling

How does variance react to the arithmetic of Session 01's linearity rules?
Two clean laws, for constants $b$ and $w$:

$$\operatorname{Var}[X + b] = \operatorname{Var}[X]
 \qquad\qquad
 \operatorname{Var}[wX] = w^2\,\operatorname{Var}[X]$$

**Proof of the shift law:** $E[X + b] = \mu + b$, so the deviation is
$(X + b) - (\mu + b) = X - \mu$ — *identical* to before. Shifting slides
every outcome and the expectation together; spread cannot change.
$\blacksquare$

**Proof of the scale law:** $E[wX] = w\mu$, so

$$\operatorname{Var}[wX] = E[(wX - w\mu)^2] = E[w^2 (X - \mu)^2]
 = w^2\,E[(X - \mu)^2] = w^2 \operatorname{Var}[X]. \;\blacksquare$$

The scale factor comes out **squared** — doubling a payout quadruples its
variance. Standard deviation, the square root, scales by $|w|$: doubling the
payout doubles $\sigma$. And the sign vanishes:
$\operatorname{Var}[-X] = \operatorname{Var}[X]$.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000).astype(np.float64)

def est_var(s):
    return ((s - s.mean()) ** 2).mean()

print("Var[X] est:      ", est_var(x), "   exact 35/12 =", 35 / 12)
print("Var[X + 10] est: ", est_var(x + 10.0))
print("Var[3X] est:     ", est_var(3.0 * x), "   9 * 35/12 =", 9 * 35 / 12)
print("Var[-X] est:     ", est_var(-x))

### Checkpoint 3

1. Temperatures in Celsius have variance $4$. Fahrenheit is
   $F = 1.8\,C + 32$. Compute $\operatorname{Var}[F]$ and $\sigma_F$ by
   the laws — which constant affected nothing, and why?
2. Find *all* values of $w$ for which $\operatorname{Var}[wX] =
   \operatorname{Var}[X]$ when $\operatorname{Var}[X] > 0$, and explain the
   physical meaning of each.

## 4. Independence

Roll two dice. Learning the first die shows 6 tells you *nothing* about the
second. That is independence, and the formal version is a statement about
the joint table: random variables $X$ and $Y$ are **independent** when

$$P(X = x \text{ and } Y = y) \;=\; P(X = x)\cdot P(Y = y)
 \qquad\text{for every pair } (x, y).$$

Every joint probability is the product of the two single probabilities — the
whole joint table factors.

**A dependent pair** for contrast: $X$ = one die, $Y = 7 - X$. Then
$P(X{=}1 \text{ and } Y{=}1) = 0$, but the product rule demands
$\tfrac16 \cdot \tfrac16 = \tfrac1{36}$. One failing pair is enough:
dependent.

In simulation, independence is *modeled* by separate draws from the
generator — each call produces fresh draws unrelated to earlier ones. We can
test the factoring numerically: for two independent dice, compare the
frequency of "first shows 6 and second shows 6" with the product of the
single frequencies.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000)
y = rng.integers(1, 7, size=200_000)     # separate draws -> independent
y_dep = 7 - x                            # determined by x -> dependent

print("independent pair:")
print("  P(X=6 and Y=6) est:    ", ((x == 6) & (y == 6)).mean())
print("  P(X=6) * P(Y=6) est:   ", (x == 6).mean() * (y == 6).mean(), "  (1/36 =", 1 / 36, ")")
print("dependent pair (Y = 7 - X):")
print("  P(X=6 and Y=6) est:    ", ((x == 6) & (y_dep == 6)).mean())
print("  P(X=6) * P(Y=6) est:   ", (x == 6).mean() * (y_dep == 6).mean())

**A stated fact we will lean on** (proof beyond this course, verification by
simulation below): *functions of independent random variables are
independent* — if $X \perp Y$, then $g(X) \perp h(Y)$ for any functions
$g, h$. Squaring the first die and doubling the second cannot create a
connection where none existed:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000)
y = rng.integers(1, 7, size=200_000)
gx, hy = x ** 2, 2 * y                   # g(X) = X^2, h(Y) = 2Y

print("P(g=36 and h=12) est: ", ((gx == 36) & (hy == 12)).mean())
print("P(g=36) * P(h=12) est:", (gx == 36).mean() * (hy == 12).mean())

### Checkpoint 4

1. $X$ is a fair coin (0 or 1) and $Y = 1 - X$. Write out all four joint
   probabilities $P(X{=}x, Y{=}y)$ and show exactly which pairs break the
   product rule.
2. Flip a coin twice (independent flips). $X$ = first flip, $S$ = total
   heads. Are $X$ and $S$ independent? Check the pair $(x, s) = (0, 2)$.

## 5. Products of independent variables: E[XY] = E[X]E[Y]

Linearity handled sums with no assumptions. Products are pickier — they need
independence.

**Claim.** If $X$ and $Y$ are independent, then $E[XY] = E[X]\,E[Y]$.

**Proof** (discrete): sum over the joint table and factor it,

$$E[XY] = \sum_x \sum_y xy\,P(X{=}x, Y{=}y)
 \overset{\text{indep}}{=} \sum_x \sum_y xy\,P(X{=}x)P(Y{=}y)
 = \Big(\sum_x x P(X{=}x)\Big)\Big(\sum_y y P(Y{=}y)\Big)
 = E[X]E[Y]. \;\blacksquare$$

The independence step is where the joint probability splits into a product,
letting the double sum factor into two separate sums.

**Without independence it fails.** Take $Y = X$ (perfectly dependent): then
$E[XY] = E[X^2]$, which Session 01 showed differs from $(E[X])^2$ whenever
$X$ has any spread at all. The gap is exactly $\operatorname{Var}[X]$.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000).astype(np.float64)
y = rng.integers(1, 7, size=200_000).astype(np.float64)

print("independent:  E[XY] est:", (x * y).mean(), "  E[X]E[Y] est:", x.mean() * y.mean())
print("dependent:    E[X*X] est:", (x * x).mean(), "  E[X]E[X] est:", x.mean() * x.mean())
print("              gap est:", (x * x).mean() - x.mean() ** 2, "  = Var[X] est (35/12 =", 35 / 12, ")")

### Checkpoint 5

1. Two independent spinners from Session 01 ($E[X] = 2.25$ each) are spun
   and their values multiplied. Compute $E[XY]$ by the product rule, then
   estimate it by seeded simulation.
2. Point to the exact step of the proof that breaks when $Y = X$. (Which
   equality used independence, and why is it false for the pair $(X, X)$?)

## 6. Variance of a sum

The payoff of Sections 2–5. For **independent** $X$ and $Y$:

$$\operatorname{Var}[X + Y] = \operatorname{Var}[X] + \operatorname{Var}[Y].$$

**Proof** (discrete, via the shortcut):

$$\begin{aligned}
\operatorname{Var}[X + Y]
 &= E[(X + Y)^2] - (E[X + Y])^2 \\
 &= E[X^2] + 2E[XY] + E[Y^2] - (E[X] + E[Y])^2
   && \text{(expand; linearity)} \\
 &= E[X^2] + 2E[X]E[Y] + E[Y^2] - E[X]^2 - 2E[X]E[Y] - E[Y]^2
   && \text{(independence: } E[XY] = E[X]E[Y]\text{)} \\
 &= \big(E[X^2] - E[X]^2\big) + \big(E[Y^2] - E[Y]^2\big)
  = \operatorname{Var}[X] + \operatorname{Var}[Y]. \;\blacksquare
\end{aligned}$$

One line used independence — the cross terms $2E[XY]$ and $-2E[X]E[Y]$
cancel only because the product rule holds. (Session 03 measures the
leftover when it doesn't: the *covariance*.)

**Where dependence wrecks it:** $\operatorname{Var}[X + X] =
\operatorname{Var}[2X] = 4\operatorname{Var}[X]$ by the scale law — not
$2\operatorname{Var}[X]$. Adding a variable *to itself* doubles every
deviation; adding an independent copy lets deviations partly cancel. Two
different physical situations, two different answers:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000).astype(np.float64)
x2 = rng.integers(1, 7, size=200_000).astype(np.float64)   # independent second die

def est_var(s):
    return ((s - s.mean()) ** 2).mean()

print("Var[X] est:            ", est_var(x), "        (35/12 ≈ 2.9167)")
print("Var[X + X'] est:       ", est_var(x + x2), "   two dice: 35/12 + 35/12 = 35/6 ≈ 5.833")
print("Var[X + X] = Var[2X]:  ", est_var(2 * x), "   4 * 35/12 = 35/3 ≈ 11.667")

By induction the sum rule extends to any number of *mutually* independent
variables: $\operatorname{Var}[X_1 + \cdots + X_n] =
\operatorname{Var}[X_1] + \cdots + \operatorname{Var}[X_n]$. For $n$
fair dice: total variance $\tfrac{35n}{12}$, standard deviation
$\sqrt{35n/12}$ — growing like $\sqrt{n}$ while the expectation grows like
$n$. That mismatch (totals grow faster than their wobble) is why casino
totals, class averages, and Session 03's bell curve are so predictable.

### Checkpoint 6

1. Sixteen independent fair dice are rolled. Compute $E$, Var, and $\sigma$
   of the total. What fraction of the expected total is one $\sigma$?
2. $X$ and $Y$ are independent with $\operatorname{Var}[X] = 3$,
   $\operatorname{Var}[Y] = 5$. Compute $\operatorname{Var}[X - Y]$.
   (Careful: it is not $-2$. Which two laws combine here?)

## 7. Scaled sums of independent factors

The exam's favorite variance derivation combines *everything* in this
session at once. Take $n$ mutually independent factors
$x_1, \dots, x_n$, each with expectation 0 and common variance $\sigma^2$,
and scale factor $w_i$ applied to each. Form the scaled sum

$$S \;=\; \sum_{i=1}^{n} w_i\,x_i .$$

**What are $E[S]$ and $\operatorname{Var}[S]$?**

*Expectation* — linearity, no independence needed:
$E[S] = \sum_i w_i E[x_i] = 0$.

*Variance* — the derivation to know cold. Since $E[S] = 0$, the shortcut
gives $\operatorname{Var}[S] = E[S^2]$. Expand the square of the sum into
$n^2$ products:

$$E[S^2] = E\Big[\sum_i \sum_j w_i w_j\, x_i x_j\Big]
 = \sum_i \sum_j w_i w_j\, E[x_i x_j].$$

Split by cases. For $i \ne j$, independence gives
$E[x_i x_j] = E[x_i]E[x_j] = 0$ — every cross term dies. For $i = j$,
$E[x_i^2] = \operatorname{Var}[x_i] = \sigma^2$ (shortcut with zero
expectation). Only the diagonal survives:

$$\boxed{\;\operatorname{Var}\Big[\sum_i w_i x_i\Big]
 = \sigma^2 \sum_{i=1}^{n} w_i^2\;}$$

With unequal variances $\sigma_i^2$ the same argument gives
$\sum_i w_i^2 \sigma_i^2$. And for $n = 1$ it collapses to Section 3's
scale law $w^2\sigma^2$ — one identity containing both.

**Worked exam-style example (reasoning required).** This is the exam's
pattern, stated generically:

> $x_1, \dots, x_{100}$ are mutually independent factors, each with
> expectation 0 and variance $\sigma^2 = 9$. All scale factors are equal:
> $w_i = c > 0$. Find the value of $c$ for which the scaled sum
> $S = c\sum_i x_i$ has $\operatorname{Var}[S] = 1$, as an exact fraction.

Step by step:

1. The identity with equal factors: $\operatorname{Var}[S] = \sigma^2
   \sum_{i} c^2 = c^2 \cdot n\,\sigma^2 = c^2 \cdot 100 \cdot 9$.
2. Demand $900\,c^2 = 1$, so $c^2 = \tfrac{1}{900}$.
3. $c > 0$: $c = \tfrac{1}{30}$. **Answer: $c = 1/30$.**

The general moral, worth memorizing in words: *to keep a sum of $n$
independent same-spread factors at unit variance, each scale factor must
shrink like $1/\sqrt{n\sigma^2}$.* Simulation check of both the identity
and the example:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

# n = 3 zero-expectation factors (fair ±1 coins scaled to variance 4: values ±2)
n_runs = 200_000
w = np.array([2.0, -1.0, 3.0])
x = rng.choice(np.array([-2.0, 2.0]), size=(n_runs, 3))    # each column: Var = 4
s = (w * x).sum(axis=1)                                    # sum_i w_i x_i, per run
print("Var[S] est:", ((s - s.mean()) ** 2).mean(),
      "  identity: sigma^2 * sum w_i^2 = 4 * 14 =", 4 * (w ** 2).sum())

# exam example: n = 100, sigma^2 = 9, c = 1/30
c = 1 / 30
x100 = rng.choice(np.array([-3.0, 3.0]), size=(n_runs, 100))   # Var = 9 each
s100 = c * x100.sum(axis=1)
print("Var[c * sum] est:", ((s100 - s100.mean()) ** 2).mean(), "  target: 1")

### Checkpoint 7

1. Independent zero-expectation factors with $\sigma^2 = 25$,
   scale factors $w = (0.6,\ 0.8)$. Compute $\operatorname{Var}[S]$ by the
   identity, then verify with a seeded ±5 coin simulation.
2. In the boxed derivation, exactly which two hypotheses killed the cross
   terms $E[x_i x_j]$, $i \ne j$? Show that dropping *either one* (keep
   independence but let $E[x_i] = m \ne 0$; or keep zero expectation but set
   all $x_i$ equal) leaves nonzero cross terms.

## 8. Common pitfalls II

**Pitfall — $E[X^2]$ vs $(E[X])^2$.** The classic. The shortcut is
$E[X^2] - (E[X])^2$, function-then-average minus average-then-function, in
that order. Wrong order → negative "variance", an immediate impossibility
(Section 2). When a computation produces a negative variance, hunt for this
swap first.

**Pitfall — adding variances without independence.**
$\operatorname{Var}[X + Y] = \operatorname{Var}[X] +
\operatorname{Var}[Y]$ *requires* independence; the sum-of-two-dice demo
and $\operatorname{Var}[X + X] = 4\operatorname{Var}[X]$ bracket the
truth. Before adding variances, ask: are these separate draws, or the same
quantity appearing twice?

**Pitfall — scaling variance linearly.** "$3X$ has three times the
variance" is wrong: $\operatorname{Var}[3X] = 9\operatorname{Var}[X]$.
The *standard deviation* scales by 3. State which of the two you are
scaling, every time.

**Pitfall — subtraction subtracts variance.** For independent $X, Y$:
$\operatorname{Var}[X - Y] = \operatorname{Var}[X] +
\operatorname{Var}[Y]$ — a plus. $-Y$ has the *same* variance as $Y$
(scale law with $w = -1$), so differencing two noisy quantities makes the
result *noisier*, never cleaner:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.integers(1, 7, size=200_000).astype(np.float64)
y = rng.integers(1, 7, size=200_000).astype(np.float64)

def est_var(s):
    return ((s - s.mean()) ** 2).mean()

print("Var[X - Y] est:", est_var(x - y), "   sum 35/12 + 35/12 =", 35 / 6)
print("(the WRONG guess Var[X] - Var[Y] would be 0)")

### Checkpoint 8

1. Each line contains one pitfall — name it and give the corrected value.
   $X, Y$ independent, each with $E = 2$ and $\operatorname{Var} = 6$:
   (i) "$\operatorname{Var}[X] = E[X]^2 - E[X^2] $";
   (ii) "$\operatorname{Var}[5X] = 30$";
   (iii) "$\operatorname{Var}[X - Y] = 0$";
   (iv) "$\operatorname{Var}[X + X] = 12$".
2. A measurement is repeated 4 times independently (each has variance
   $\sigma^2$) and the results are *averaged*. Show the average's variance
   is $\sigma^2/4$ — which two laws combine, and in what order?

## Checkpoint answers

### Checkpoint 1 answers

In [ ]:
# 1. Fair coin: mu = 0, Var = (1/2)(1) + (1/2)(1) = 1, sigma = 1.
#    Rigged coin: mu = 0.9(-1) + 0.1(9) = 0. Var = 0.9(0 - (-1))^2 ... careful:
#    deviations from mu = 0: Var = 0.9(-1)^2 + 0.1(9)^2 = 0.9 + 8.1 = 9,
#    sigma = 3.
# 2. E[X - E[X]] = E[X] - E[X] = 0 by linearity — always, so it ranks every
#    variable as equally "spread". E[|X - E[X]|] for game A:
#    (1/2)|-1| + (1/2)|1| = 1 (sizes cannot cancel). The squared version is
#    standard because squares have the clean algebra of Sections 2-7.
print("see comments")

### Checkpoint 2 answers

In [ ]:
# 1. Var[X] = E[X^2] - E[X]^2 = 91/6 - 49/4 = 182/12 - 147/12 = 35/12.
print(91 / 6 - 3.5 ** 2, 35 / 12)

# 2. Variance is E[(X - mu)^2], an average of squares, so it is never
#    negative — a negative result means the subtraction was reversed:
#    it computed E[X]^2 - E[X^2] instead of E[X^2] - E[X]^2.

### Checkpoint 3 answers

In [ ]:
# 1. Var[F] = 1.8^2 * 4 = 12.96, sigma_F = 3.6 (= 1.8 * 2).
#    The +32 shift affected nothing: shifting slides outcomes and expectation
#    together, leaving every deviation unchanged.
print(1.8 ** 2 * 4, np.sqrt(1.8 ** 2 * 4))

# 2. w^2 Var[X] = Var[X]  =>  w^2 = 1  =>  w = 1 or w = -1.
#    w = 1 leaves X alone; w = -1 mirrors every outcome about 0 — the
#    spread of outcomes is unchanged, only their signs flip.

### Checkpoint 4 answers

In [ ]:
# 1. P(0,1) = 1/2, P(1,0) = 1/2, P(0,0) = 0, P(1,1) = 0.
#    Products are 1/4 for ALL four pairs, so every pair breaks the rule —
#    e.g., P(X=0, Y=0) = 0 but P(X=0)P(Y=0) = 1/4.
# 2. Not independent. P(X=0 and S=2) = 0 (a 2-head total needs the first
#    flip to be heads), but P(X=0) * P(S=2) = (1/2)(1/4) = 1/8 > 0.
print("see comments")

### Checkpoint 5 answers

In [ ]:
# 1. E[XY] = E[X]E[Y] = 2.25 * 2.25 = 5.0625 (independent spinners).
SEED = 20260804
rng = np.random.default_rng(SEED)
values = np.array([1.0, 2.0, 5.0])
probs = np.array([0.5, 0.25, 0.25])
a = rng.choice(values, size=200_000, p=probs)
b = rng.choice(values, size=200_000, p=probs)
print("E[XY] est:", (a * b).mean(), "  exact:", 2.25 ** 2)

# 2. The step P(X=x, Y=y) = P(X=x)P(Y=y). For the pair (X, X) the joint
#    probability of (x, x') is 0 whenever x != x' — nothing factors; the
#    double sum collapses to the diagonal and E[X·X] = E[X^2] != E[X]^2.

### Checkpoint 6 answers

In [ ]:
# 1. E = 16 * 3.5 = 56;  Var = 16 * 35/12 = 140/3 ≈ 46.67;
#    sigma = sqrt(140/3) ≈ 6.83  ->  about 12% of the expected total.
print(16 * 3.5, 16 * 35 / 12, np.sqrt(16 * 35 / 12))

# 2. Var[X - Y] = Var[X] + Var[(-1)Y] = Var[X] + (-1)^2 Var[Y] = 3 + 5 = 8.
#    Scale law (w = -1) first, then the independent-sum law.

### Checkpoint 7 answers

In [ ]:
# 1. Var[S] = sigma^2 * (w1^2 + w2^2) = 25 * (0.36 + 0.64) = 25.
SEED = 20260804
rng = np.random.default_rng(SEED)
w = np.array([0.6, 0.8])
x = rng.choice(np.array([-5.0, 5.0]), size=(200_000, 2))    # Var = 25 each
s = (w * x).sum(axis=1)
print("Var[S] est:", ((s - s.mean()) ** 2).mean(), "  exact: 25")

# 2. Independence (to factor E[x_i x_j] into E[x_i]E[x_j]) AND zero
#    expectation (to make that product 0).
#    - Independent but E[x_i] = m != 0: cross terms E[x_i]E[x_j] = m^2 > 0.
#    - Zero expectation but all equal (x_i = x_1): E[x_i x_j] = E[x_1^2] =
#      sigma^2 != 0 — nothing factors, every cross term survives.

### Checkpoint 8 answers

In [ ]:
# 1. (i)  Reversed shortcut — Var[X] = E[X^2] - E[X]^2 (here: 6).
#    (ii) Linear scaling of variance — Var[5X] = 25 * 6 = 150 (30 would be
#         right for 5*sigma^2 only if sigma^2 scaled linearly; it doesn't).
#    (iii) Subtraction pitfall — Var[X - Y] = 6 + 6 = 12 for independent X, Y.
#    (iv) Missing dependence — X + X = 2X, so Var = 4 * 6 = 24, not 12
#         (12 would be the answer for two INDEPENDENT copies).

# 2. The average is A = (1/4)(X1 + X2 + X3 + X4).
#    Sum law first (independence): Var[X1+...+X4] = 4 sigma^2.
#    Then scale law: Var[A] = (1/4)^2 * 4 sigma^2 = sigma^2 / 4.
#    (Averaging independent repeats shrinks variance — the measured mean is
#    more reliable than any single measurement.)
print("see comments")